[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/04-how-a-model-learns.ipynb)

# Module 4: How a Model Learns From Data

Companion notebook to [Module 4](https://transformers-and-math.drmhse.com/modules/04-how-a-model-learns) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Part 1 — Without a nonlinearity, depth adds nothing (section 4.2)

In [ ]:
W1, W2 = torch.randn(4, 16), torch.randn(16, 3)
x = torch.randn(5, 4)
print('two linear layers == one:', torch.allclose(x @ W1 @ W2, x @ (W1 @ W2), atol=1e-5))
print('rank of the combined matrix:', torch.linalg.matrix_rank(W1 @ W2).item())

## Part 2 — Softmax, overflow and the cross-entropy gradient (section 4.3)

In [ ]:
z = torch.tensor([120., 0., -120.])
naive = torch.exp(z) / torch.exp(z).sum()
safe = torch.exp(z - z.max()) / torch.exp(z - z.max()).sum()
print('naive softmax:', naive, '\nsafe softmax: ', safe)
print('fp16 exp(12):', torch.exp(torch.tensor(12.0, dtype=torch.float16)))

logits = torch.tensor([2.0, 1.0, 0.0], requires_grad=True)
loss = F.cross_entropy(logits[None], torch.tensor([0]))
loss.backward()
print('gradient:', logits.grad, ' = softmax − one-hot:', torch.softmax(logits, 0).detach() - torch.tensor([1., 0., 0.]))

## Part 3 — Data: two interleaved spirals, split into training and validation (section 4.4)

In [ ]:
def spirals(n, seed, noise=0.09):
    g = torch.Generator().manual_seed(seed)
    label = torch.arange(n) % 2
    t = 0.4 + 2.6 * torch.rand(n, generator=g)
    a = t * 2.2 + label * math.pi
    pts = torch.stack([t / 3 * torch.cos(a), t / 3 * torch.sin(a)], 1) + noise * torch.randn(n, 2, generator=g)
    return pts, label

Xval, yval = spirals(300, 99)
Xs, ys = spirals(30, 7)
Xl, yl = spirals(400, 7)
plt.scatter(*Xl.T, c=yl, cmap='coolwarm', s=8); plt.title('training data (400 points)'); plt.show()

## Part 4 — Train an MLP, and watch it overfit (sections 4.4 and 4.7)
The same network on 30 and on 400 training points. Everything else is held fixed: an ablation (section 4.6).

In [ ]:
def train(X, y, epochs=300, width=64, seed=0, weight_decay=0.0):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(2, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU(), nn.Linear(width, 2))
    opt = torch.optim.SGD(net.parameters(), lr=0.1, weight_decay=weight_decay)
    hist = []
    for epoch in range(epochs):
        for idx in torch.randperm(len(X)).split(16):
            opt.zero_grad()
            F.cross_entropy(net(X[idx]), y[idx]).backward()
            opt.step()
        with torch.no_grad():
            hist.append((F.cross_entropy(net(X), y).item(), F.cross_entropy(net(Xval), yval).item()))
    return net, hist

for X, y, label in [(Xs, ys, '30 points'), (Xl, yl, '400 points')]:
    net, hist = train(X, y)
    tr, va = zip(*hist)
    plt.plot(tr, label=f'{label}: train'); plt.plot(va, '--', label=f'{label}: validation')
    print(f'{label}: final train {tr[-1]:.3f}, final val {va[-1]:.3f}, best val {min(va):.3f} at epoch {va.index(min(va)) + 1}')
plt.ylim(0, 1.5); plt.legend(); plt.xlabel('epoch'); plt.show()

## Part 5 — Seeds and run-to-run noise (section 4.6)
Repeat the 400-point run with three seeds. A difference smaller than this spread is not a result.

In [ ]:
finals = [train(Xl, yl, epochs=100, seed=s)[1][-1][1] for s in range(3)]
print('validation loss over seeds:', [round(v, 4) for v in finals], ' spread:', round(max(finals) - min(finals), 4))

## Your turn
1. Add `weight_decay=1e-3` to the 30-point run. Does the gap between training and validation loss shrink?
2. Replace `nn.ReLU()` with `nn.Identity()`. What happens to the validation loss, and why (section 4.2)?

<details><summary>Hints</summary>

1. Compare the final training and validation losses with and without decay. 2. Without a nonlinearity the network can only draw a straight boundary, which cannot separate spirals: both losses stay near $\ln 2 \approx 0.69$.
</details>